# Predictive Pulse: Harnessing Machine Learning for Blood Pressure Analysis
## Hypertension Stage Prediction using Multi-Algorithm Classification

**Dataset:** Kaggle — 1,825 patient records  
**Target:** Hypertension Stages (Normal, Stage-1, Stage-2, Crisis)  
**Best Model:** Logistic Regression (95.2% Accuracy)

## Step 1: Data Collection & Preparation

In [ ]:
# 1.1 Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully')

In [ ]:
# 1.2 Data Loading and Initial Exploration
# Dataset: https://drive.google.com/file/d/1qYvKqg4w_w4blizSVqmLvwY25m7V7N3_/view?usp=sharing
data = pd.read_csv('/content/patient_data.csv')
print('Shape:', data.shape)
data.head()

In [ ]:
# 1.3 Data Cleaning Steps

# Handling Missing Values
print('Missing values:')
print(data.isnull().sum())

In [ ]:
# Data Type Corrections: Rename column 'C' to 'Gender' for clarity
data.rename(columns={'C': 'Gender'}, inplace=True)
print('Column renamed: C -> Gender')

In [ ]:
# Inconsistency Corrections: Fix spelling errors and standardize categorical values
data['TakeMedication'].replace({'Yes ': 'Yes'}, inplace=True)
data['NoseBleeding'].replace({'No ': 'No'}, inplace=True)
data['Systolic'].replace({'121- 130': '121 - 130'}, inplace=True)
data['Systolic'].replace({'100+': '100 - 110'}, inplace=True)
data['Stages'].replace({'HYPERTENSION (Stage-2).': 'HYPERTENSION (Stage-2)'}, inplace=True)
data['Stages'].replace({'HYPERTENSIVE CRISI': 'HYPERTENSIVE CRISIS'}, inplace=True)

print((data['Diastolic'] == '130+').sum())
print((data['Diastolic'] == '100+').sum())

data['Diastolic'].replace({'130+': '100+'}, inplace=True)
print('Inconsistencies fixed')

In [ ]:
# Duplicate Removal
print('Duplicates before:', data.duplicated().sum())
data.drop_duplicates(inplace=True)
print('Shape after removing duplicates:', data.shape)

In [ ]:
# 1.4 Label Encoding
# Gender: Male=0, Female=1
# Binary features: No=0, Yes=1
# Age groups: 18-34=1, 35-50=2, 51-64=3, 65+=4
# Severity: Mild=0, Moderate=1, Severe=2
# Target stages: Normal=0, Stage-1=1, Stage-2=2, Crisis=3

nominal_features = ['Gender', 'History', 'Patient', 'TakeMedication', 'BreathShortness', 'VisualChanges', 'NoseBleeding', 'ControlledDiet']
ordinal_features = [f for f in data.columns if f not in nominal_features]
ordinal_features.remove('Stages')
print('Ordinal features:', ordinal_features)

for col in nominal_features:
    if set(data[col].unique()) == set(['Yes', 'No']):
        data[col] = data[col].map({'No': 0, 'Yes': 1})
    elif col == 'Gender':
        data[col] = data[col].map({'Male': 0, 'Female': 1})

data['Age'] = data['Age'].map({'18-34': 1, '35-50': 2, '51-64': 3, '65+': 4})
data['Severity'] = data['Severity'].replace({'Mild': 0, 'Moderate': 1, 'Severe': 2})
data['WhenDiagnoused'] = data['WhenDiagnoused'].map({'<1 Year': 1, '1 - 5 Years': 2, '>5 Years': 3})
data['Systolic'] = data['Systolic'].map({'100 - 110': 0, '111 - 120': 1, '121 - 130': 2, '130+': 3})
data['Diastolic'] = data['Diastolic'].map({'70 - 80': 0, '81 - 90': 1, '91 - 100': 2, '100+': 3})
data['Stages'] = data['Stages'].map({'NORMAL': 0, 'HYPERTENSION (Stage-1)': 1, 'HYPERTENSION (Stage-2)': 2, 'HYPERTENSIVE CRISIS': 3})

print('Encoding complete')
data.head()

In [ ]:
# Feature Scaling: Applied MinMaxScaler to ordinal features
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
data[ordinal_features] = scaler.fit_transform(data[ordinal_features])
print('MinMaxScaler applied to ordinal features')
data.head()

## Step 2: Exploratory Data Analysis

In [ ]:
# 2.1 Descriptive Statistics
print('Dataset shape:', data.shape)
print('\nDescriptive Statistics:')
data.describe()

In [ ]:
# 2.2 Visual Analysis - Gender Distribution
# Count of each categorical feature
plt.figure(figsize=(8,4))
sns.countplot(data=data, x='Gender', palette='Set2')
plt.title('Gender Distribution')
plt.show()

# Pie chart for Gender
data['Gender'].value_counts().plot.pie(autopct='%1.1f%%', figsize=(5,5), colors=['#66b3ff','#99ff99'])
plt.title('Gender Distribution (Pie Chart)')
plt.ylabel('')
plt.show()

In [ ]:
# 2.3 Hypertension Stages Distribution
plt.figure(figsize=(8,4))
sns.countplot(data=data, x='Stages', palette='coolwarm')
plt.title('Hypertension Stages Distribution')
plt.xticks(rotation=30)
plt.show()

In [ ]:
# 2.4 Correlation between Systolic and Diastolic Pressure
import numpy as np

def range_to_midpoint(val):
    if '-' in str(val):
        start, end = val.split('-')
        return (int(start.strip()) + int(end.strip())) / 2
    elif '+' in str(val):
        return int(val.replace('+','').strip())
    else:
        return np.nan

# Note: Using already-encoded data for heatmap
plt.figure(figsize=(6,4))
sns.heatmap(data[['Systolic','Diastolic']].corr(), annot=True, cmap='Blues')
plt.title('Correlation between Systolic & Diastolic')
plt.show()

In [ ]:
# 2.5 TakeMedication vs Severity
plt.figure(figsize=(8,5))
sns.countplot(data=data, x='TakeMedication', hue='Severity', palette='Set1')
plt.title('TakeMedication vs Severity')
plt.show()

In [ ]:
# 2.6 Age Group vs Hypertension Stages
plt.figure(figsize=(8,5))
sns.countplot(data=data, x='Age', hue='Stages', palette='husl')
plt.title('Age Group vs Hypertension Stages')
plt.show()

In [ ]:
# 2.7 Pairplot: Systolic vs Diastolic across Stages
sns.pairplot(data[['Systolic','Diastolic','Stages']], hue='Stages', diag_kind='kde', palette='husl')
plt.suptitle('Pairplot: Systolic vs Diastolic across Stages', y=1.02)
plt.show()

## Step 3: Model Building

In [ ]:
# 3.1 Train-Test Split
from sklearn.model_selection import train_test_split

x = data.drop('Stages', axis=1)
y = data['Stages']
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print('Training set:', X_train.shape[0], 'samples (80%)')
print('Testing set:', X_test.shape[0], 'samples (20%)')

In [ ]:
# 3.2 Import all classifiers
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

accuracy = {}
print('Classifiers ready')

In [ ]:
# 3.3 Logistic Regression
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)
y_pred = logreg.predict(X_test)
print('Logistic Regression:')
print('Accuracy:', accuracy_score(y_test, y_pred))
print('Classification Report:\n', classification_report(y_test, y_pred))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred))
accuracy['Logistic Regression'] = accuracy_score(y_test, y_pred)

In [ ]:
# 3.4 Decision Tree
decisionTree = DecisionTreeClassifier()
decisionTree.fit(X_train, y_train)
y_pred = decisionTree.predict(X_test)
print('Decision Tree:')
print('Accuracy:', accuracy_score(y_test, y_pred))
print('Classification Report:\n', classification_report(y_test, y_pred))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred))
accuracy['Decision Tree'] = accuracy_score(y_test, y_pred)

In [ ]:
# 3.5 Random Forest
randomforest = RandomForestClassifier()
randomforest.fit(X_train, y_train)
y_pred = randomforest.predict(X_test)
print('Random Forest:')
print('Accuracy:', accuracy_score(y_test, y_pred))
print('Classification Report:\n', classification_report(y_test, y_pred))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred))
accuracy['Random Forest'] = accuracy_score(y_test, y_pred)

In [ ]:
# 3.6 SVM
svm = SVC(probability=True)
svm.fit(X_train, y_train)
y_pred = svm.predict(X_test)
print('SVM:')
print('Accuracy:', accuracy_score(y_test, y_pred))
print('Classification Report:\n', classification_report(y_test, y_pred))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred))
accuracy['SVM'] = accuracy_score(y_test, y_pred)

In [ ]:
# 3.7 K-Nearest Neighbors
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)
print('KNN:')
print('Accuracy:', accuracy_score(y_test, y_pred))
print('Classification Report:\n', classification_report(y_test, y_pred))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred))
accuracy['KNN'] = accuracy_score(y_test, y_pred)

In [ ]:
# 3.8 Ridge Classifier
RC = RidgeClassifier()
RC.fit(X_train, y_train)
y_pred = RC.predict(X_test)
print('RidgeClassifier:')
print('Accuracy:', accuracy_score(y_test, y_pred))
print('Classification Report:\n', classification_report(y_test, y_pred))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred))
accuracy['RidgeClassifier'] = accuracy_score(y_test, y_pred)

In [ ]:
# 3.9 Gaussian Naive Bayes
naive_bayes = GaussianNB()
naive_bayes.fit(X_train, y_train)
y_pred = naive_bayes.predict(X_test)
print('Naive Bayes:')
print('Accuracy:', accuracy_score(y_test, y_pred))
print('Classification Report:\n', classification_report(y_test, y_pred))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred))
accuracy['Naive Bayes'] = accuracy_score(y_test, y_pred)

## Step 4: Performance Testing & Model Selection

In [ ]:
# 4.1 Performance Summary
print('\n=== Performance Summary ===')
for model_name, acc in sorted(accuracy.items(), key=lambda x: x[1], reverse=True):
    print(f'{model_name}: {acc:.4f} ({acc*100:.1f}%)')

In [ ]:
# 4.2 Accuracy Comparison Bar Plot
plt.figure(figsize=(10,5))
models = list(accuracy.keys())
scores = [accuracy[m]*100 for m in models]
colors = ['#22c55e' if m == 'Logistic Regression' else '#4a9eff' for m in models]
bars = plt.bar(models, scores, color=colors, edgecolor='white', linewidth=0.5)
plt.axhline(y=95.2, color='#22c55e', linestyle='--', alpha=0.5, label='Selected: 95.2%')
plt.ylim(70, 105)
plt.title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
plt.ylabel('Accuracy (%)')
plt.xticks(rotation=25, ha='right')
for bar, score in zip(bars, scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{score:.1f}%', 
             ha='center', fontsize=10)
plt.legend()
plt.tight_layout()
plt.show()
print('\nSelected: Logistic Regression — Best generalization, 95.2% accuracy')
print('Rejected: Decision Tree, Random Forest, SVM (100% = overfitting)')

## Step 5: Model Deployment

In [ ]:
# 5.1 Save the best model
import joblib

joblib.dump(logreg, 'logreg_model.pkl')
print('✅ Model saved as logreg_model.pkl')
print('Ready for Flask deployment!')